# Credits
* I forked [thedrcat's](https://www.kaggle.com/thedrcat) both [train](https://www.kaggle.com/code/thedrcat/predict-with-daigt-v3-train-dataset) and [inference](https://www.kaggle.com/code/thedrcat/detectai-roberta-large-argugpt-infer) notebooks for this DistilRoberta notebook.
* Also used as datasets:
  * '/kaggle/input/daigt-external-dataset/daigt_external_dataset.csv'
  * "/kaggle/input/llm-7-prompt-training-dataset/train_essays_RDizzl3_seven_v1.csv"
  * "/kaggle/input/feedback21-and-fb-effectiveness/feedback-prize-2021.csv"
  * "/kaggle/input/feedback21-and-fb-effectiveness/feedback-prize-effectiveness.csv"
  * "/kaggle/input/h2oai-predict-the-llm/train.csv"
  * "/kaggle/input/daigt-v2-train-dataset/train_v2_drcat_02.csv"
  * "/kaggle/input/llm-generated-essay-using-palm-from-google-gen-ai/LLM_generated_essay_PaLM.csv"
  * "/kaggle/input/llm-mistral-7b-instruct-texts/Mistral7B_CME_v7_15_percent_corruption.csv"
  * "/kaggle/input/daigt-data-llama-70b-and-falcon180b/llama_falcon_v3.csv"
  * "/kaggle/input/hello-claude-1000-essays-from-anthropic/persuade15_claude_instant1.csv"
 
 I thank all of you. If I forget anything, I am sorry.
 
# Attention

**This training dataset has been created on a trial-and-error basis. I am unsure of its performance in private tests regarding data or label leakage. Please use it <u>cautiously</u>, in blending etc.**

In [ ]:
import transformers
import datasets
import pandas as pd
import numpy as np
from datasets import Dataset
import os

In [ ]:
model_checkpoint = "/kaggle/input/deberta-v3-large" #base model

In [ ]:
df_ai = pd.read_csv('/kaggle/input/daigt-external-dataset/daigt_external_dataset.csv')

In [ ]:
df_ai = pd.read_csv('/kaggle/input/daigt-external-dataset/daigt_external_dataset.csv')
print(len(df_ai))
df_ai.head()
df_ai = df_ai.sample(frac=1, random_state=42)
df_train = df_ai[:-200]
df_valid = df_ai[-200:]
t1, t2 = pd.DataFrame(), pd.DataFrame()
t1['text'] = df_train.source_text
t1['label'] = 0
t2['text'] = df_train.text
t2['label'] = 1
train = pd.concat([t1,t2]).sample(frac=1).reset_index(drop=True)
v1, v2 = pd.DataFrame(), pd.DataFrame()
v1['text'] = df_valid.source_text
v1['label'] = 0
v2['text'] = df_valid.text
v2['label'] = 1
valid = pd.concat([v1,v2]).sample(frac=1).reset_index(drop=True)

In [ ]:
llm7prompt = pd.read_csv("/kaggle/input/llm-7-prompt-training-dataset/train_essays_RDizzl3_seven_v1.csv")
llm7prompt["chars_count"] = llm7prompt.text.apply(len)

In [ ]:
feedback_1 = pd.read_csv("/kaggle/input/feedback21-and-fb-effectiveness/feedback-prize-2021.csv")
feedback_1["chars_count"]=feedback_1.text.apply(len)
feedback_2 = pd.read_csv("/kaggle/input/feedback21-and-fb-effectiveness/feedback-prize-effectiveness.csv")
feedback_2["chars_count"]=feedback_2.text.apply(len)
A = feedback_1[(feedback_1["chars_count"]>1000 )& (feedback_1["chars_count"]<1600)][["text"]]
B = feedback_2[(feedback_2["chars_count"]>1000 )& (feedback_2["chars_count"]<1600)][["text"]]
C = pd.concat([A,B],axis=0).reset_index(drop=True)
C["label"] = 0

In [ ]:
h2o = pd.read_csv("/kaggle/input/h2oai-predict-the-llm/train.csv")[np.invert(pd.read_csv("/kaggle/input/h2oai-predict-the-llm/train.csv").Response.isnull())].rename(columns={"Response":"text"})[["text"]]
h2o["label"] = 1
h2o.text = h2o.text.fillna("")

In [ ]:
llm7prompt.label = llm7prompt.label.apply(lambda x: 1 if x==0 else 0  )

In [ ]:
train2 = pd.read_csv("/kaggle/input/daigt-v2-train-dataset/train_v2_drcat_02.csv", sep=',')
train2 = train2.drop_duplicates(subset=['text'])

train2.label = train2.label.apply(lambda x: 1 if x==0 else 0)

In [ ]:
train0 = pd.concat([
    llm7prompt,
    train ,
    train2,
                   C,
                    h2o,

                   valid],
    
    axis=0).reset_index(drop=True).drop_duplicates()\
                                                .sample(frac=1, random_state=42)\
                                                    .reset_index(drop=True)[["text",
      "label"]]
train0 = pd.concat([
    llm7prompt,
    
    train ,
    train2,
                   C,
                    h2o,

                   valid],
    
    axis=0).reset_index(drop=True).drop_duplicates()\
                                                .sample(frac=1, random_state=42)\
                                                    .reset_index(drop=True)[["text",
      "label"]].iloc[train0.text.drop_duplicates().index].reset_index(drop=True)

In [ ]:
palm = pd.read_csv("/kaggle/input/llm-generated-essay-using-palm-from-google-gen-ai/LLM_generated_essay_PaLM.csv")[["text"]]
palm["label"] = 0
mistral = pd.read_csv("/kaggle/input/llm-mistral-7b-instruct-texts/Mistral7B_CME_v7_15_percent_corruption.csv")[["text"]]
mistral["label"] = 0
llama = pd.read_csv("/kaggle/input/daigt-data-llama-70b-and-falcon180b/llama_falcon_v3.csv")[["text"]]
llama["label"] = 0
claude = pd.read_csv("/kaggle/input/hello-claude-1000-essays-from-anthropic/persuade15_claude_instant1.csv")[["essay_text"]]
claude["label"] = 0
claude.rename(columns={"essay_text":"text"},inplace=True)
train0 = pd.concat([train0,claude,mistral,llama,palm],axis=0).sample(frac=1).reset_index(drop=True)

In [ ]:
train0.label.value_counts()

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
sk = StratifiedKFold(n_splits=10,shuffle=True,random_state=42)

In [ ]:
for i, (tr,val) in enumerate(sk.split(train0,train0.label)):
    train = train0.iloc[tr]
    valid = train0.iloc[val]
    break

In [ ]:
train.label.value_counts()

In [ ]:
valid.label.value_counts()

In [ ]:
train.text = train.text.fillna("")
valid.text = valid.text.apply(lambda x: x.strip('\n'))
train.text = train.text.apply(lambda x: x.strip('\n'))

In [ ]:
ds_train = Dataset.from_pandas(train)
ds_valid = Dataset.from_pandas(valid)

In [ ]:
from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
def preprocess_function(examples):
    return tokenizer(examples['text'], max_length=128, padding=True, truncation=True)

ds_train_enc = ds_train.map(preprocess_function, batched=True)

ds_valid_enc = ds_valid.map(preprocess_function, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Move your model and data to the GPU
model.to(device);

from transformers import EarlyStoppingCallback

early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

num_train_epochs=16.0

In [ ]:
metric_name = "roc_auc"
model_name = "distilroberta"#"deberta-large"
batch_size = 2

args = TrainingArguments(
    f"{model_name}-finetuned_v5",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    lr_scheduler_type = "cosine",
    
    optim="adamw_torch",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=8,
    num_train_epochs=num_train_epochs,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    report_to='none',
    save_total_limit=2,
    
)

In [ ]:
from sklearn.metrics import roc_auc_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = np.exp(logits) / np.sum(np.exp(logits), axis=-1, keepdims=True)
    auc = roc_auc_score(labels, probs[:,1], multi_class='ovr')
    return {"roc_auc": auc}

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=ds_train_enc,
    eval_dataset=ds_valid_enc,
    tokenizer=tokenizer,
    callbacks = [early_stopping],
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()